# Fabric Workspace Inventory v3.2 – Production Notebook

### Data sources

| API | Purpose | Permission |
|-----|---------|------------|
| Core Items API | Item list (name, type, ID) | Viewer |
| Admin Items API | Owner, lastUpdatedDate, state | Fabric Admin |
| Scanner API (semantic-link-labs) | Created date, modified by | Fabric Admin |
| Power BI Activity Events API | Genuine last-used / access data | Fabric Admin |
| Unused Artifacts (semantic-link-labs) | Unused detection + created/last-accessed dates | Fabric Admin |

> **`Last Modified ≠ Last Used`** — This notebook keeps these strictly separate.


## 1. Install dependencies

In [ ]:
try:
    import sempy_labs
    print("semantic-link-labs is available.")
except ImportError:
    print("Installing semantic-link-labs...")
    %pip python -m pip install --upgrade pip
    %pip install semantic-link-labs -q
    print("Installed. If the kernel restarted, re-run all cells from here.")


## 2. Parameters

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PARAMETERS
# ═══════════════════════════════════════════════════════════════

workspace_id             = ""
save_to_lakehouse        = True
lakehouse_table_name     = "workspace_inventory_snapshot"
stale_cutoff_days        = 90
activity_lookback_days   = 30
enable_scanner_api       = True
enable_unused_artifacts  = True
enable_activity_events   = True


## 3. Setup and authentication

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import uuid
import logging
from datetime import datetime, timezone, timedelta
import notebookutils

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("fabric_inventory")

FABRIC_API_BASE  = "https://api.fabric.microsoft.com/v1"
POWERBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"
FABRIC_APP       = "https://app.fabric.microsoft.com"

snapshot_id       = str(uuid.uuid4())
snapshot_time_utc = datetime.now(timezone.utc).isoformat()
log.info(f"Snapshot ID: {snapshot_id}")
log.info(f"Snapshot time: {snapshot_time_utc}")

section_times = {}
def timed_section(name):
    class _Timer:
        def __enter__(self):
            self.start = time.time()
            log.info(f"▶ Starting: {name}")
            return self
        def __exit__(self, *args):
            elapsed = time.time() - self.start
            section_times[name] = elapsed
            log.info(f"✔ Completed: {name} ({elapsed:.1f}s)")
    return _Timer()


In [ ]:
with timed_section("Authentication"):
    if not workspace_id:
        workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    log.info(f"Target workspace: {workspace_id}")

    token = notebookutils.credentials.getToken("pbi")
    token_acquired_at = time.time()

    def get_headers():
        global token, token_acquired_at
        if time.time() - token_acquired_at > 2400:
            log.info("Refreshing authentication token...")
            token = notebookutils.credentials.getToken("pbi")
            token_acquired_at = time.time()
        return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    log.info("Authentication successful.")


## 4. API helpers

In [ ]:
def call_fabric_api(url, max_retries=5, caller="API"):
    for attempt in range(max_retries):
        resp = requests.get(url, headers=get_headers())
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            log.warning(f"[{caller}] Throttled. Waiting {wait}s (attempt {attempt+1})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            raise PermissionError(f"[{caller}] Access denied ({resp.status_code}): {resp.text[:300]}")
        else:
            log.error(f"[{caller}] HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
    raise RuntimeError(f"[{caller}] Failed after {max_retries} retries")


def call_powerbi_api(url, max_retries=5, caller="API"):
    for attempt in range(max_retries):
        resp = requests.get(url, headers=get_headers())
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            log.warning(f"[{caller}] Throttled. Waiting {wait}s (attempt {attempt+1})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            raise PermissionError(f"[{caller}] Access denied ({resp.status_code}): {resp.text[:300]}")
        else:
            log.error(f"[{caller}] HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
    raise RuntimeError(f"[{caller}] Failed after {max_retries} retries")


## 5. Workspace metadata

In [ ]:
with timed_section("Workspace metadata"):
    workspace_name = ""
    try:
        ws_info = call_fabric_api(f"{FABRIC_API_BASE}/workspaces/{workspace_id}", caller="Workspace")
        workspace_name = ws_info.get("displayName", "")
        log.info(f"Workspace name: {workspace_name}")
    except Exception as e:
        log.warning(f"Could not retrieve workspace name: {e}")


## 6. Layer 1 — Core Items API

In [ ]:
with timed_section("Core Items API"):
    def get_core_items(ws_id):
        items, url = [], f"{FABRIC_API_BASE}/workspaces/{ws_id}/items"
        while url:
            data = call_fabric_api(url, caller="CoreItems")
            items.extend(data.get("value", []))
            cont = data.get("continuationToken")
            url = (f"{FABRIC_API_BASE}/workspaces/{ws_id}/items?continuationToken={cont}") if cont else None
        return items

    core_items = get_core_items(workspace_id)
    log.info(f"Core Items API returned {len(core_items)} items.")

    if core_items:
        df_core = pd.DataFrame(core_items)
        df_core = df_core[["id", "displayName", "type", "description"]].rename(columns={"displayName": "name"})
    else:
        df_core = pd.DataFrame(columns=["id", "name", "type", "description"])


## 7. Layer 2 — Admin Items API

In [ ]:
admin_available = False
df_admin = pd.DataFrame()
admin_only_ids = set()
admin_items_raw = []

with timed_section("Admin Items API"):
    try:
        def extract_principal(principal_obj):
            if not principal_obj or not isinstance(principal_obj, dict):
                return None
            user = principal_obj.get("userDetails", {})
            return user.get("userPrincipalName") or principal_obj.get("displayName") or principal_obj.get("id")

        def get_admin_items(ws_id):
            items, url = [], f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}"
            while url:
                data = call_fabric_api(url, caller="AdminItems")
                items.extend(data.get("itemEntities", []))
                cont = data.get("continuationToken")
                url = (f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}&continuationToken={cont}") if cont else None
            return items

        admin_items_raw = get_admin_items(workspace_id)
        log.info(f"Admin Items API returned {len(admin_items_raw)} items.")
        admin_available = True

        if admin_items_raw:
            log.info(f"Admin API sample keys: {list(admin_items_raw[0].keys())}")

        df_admin = pd.DataFrame([{
            "id":            i.get("id"),
            "created_by":    extract_principal(i.get("creatorPrincipal")),
            "last_modified": i.get("lastUpdatedDate"),
            "state":         i.get("state"),
            "capacity_id":   i.get("capacityId"),
        } for i in admin_items_raw])

        admin_only_ids = set(df_admin["id"].dropna()) - set(df_core["id"].dropna())
        if admin_only_ids:
            log.info(f"{len(admin_only_ids)} admin-only items (system-generated).")

    except PermissionError as e:
        log.warning(f"Admin API not accessible: {e}")
        log.info("Continuing with Core Items only.")


## 8. Scanner API — created_date and modified_by

`scan_workspaces()` returns a **dict** (workspace scan result), not a DataFrame.
Items are nested under keys like `datasets`, `reports`, etc.
Each item may contain `createdDateTime` and `configuredBy`.


In [ ]:
scanner_created = {}
scanner_modified = {}

with timed_section("Scanner API (created_date, modified_by)"):
    if not enable_scanner_api:
        log.info("Scanner API disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            import sempy_labs.admin as sll_admin

            log.info("Calling scan_workspaces()...")
            scan_result = None

            try:
                scan_result = sll_admin.scan_workspaces(workspace=workspace_id)
            except TypeError:
                pass

            if scan_result is None:
                try:
                    scan_result = sll_admin.scan_workspaces()
                except Exception as e2:
                    log.warning(f"scan_workspaces() failed: {e2}")

            if scan_result is None:
                log.warning("scan_workspaces() returned None.")

            elif isinstance(scan_result, dict):
                log.info(f"Scanner returned dict. Top keys: {list(scan_result.keys())[:15]}")

                # ── Navigate: result → workspaces[0] → per-type item lists ──
                # Structure: {"workspaces": [{"id":..., "datasets":[...], "reports":[...], ...}]}
                workspaces_list = scan_result.get("workspaces", [])
                if not workspaces_list:
                    # Maybe the dict itself IS the workspace (no "workspaces" wrapper)
                    workspaces_list = [scan_result]

                target_ws = None
                for ws in workspaces_list:
                    if isinstance(ws, dict):
                        ws_id = str(ws.get("id", ""))
                        if ws_id == workspace_id or len(workspaces_list) == 1:
                            target_ws = ws
                            break

                if not target_ws:
                    log.warning("Could not find target workspace in Scanner API response.")
                else:
                    log.info(f"  Workspace keys: {list(target_ws.keys())[:20]}")
                    items_found = 0

                    # Iterate over every key in the workspace dict.
                    # Keys that hold lists of dicts are item-type collections.
                    for type_key, type_items in target_ws.items():
                        if not isinstance(type_items, list) or len(type_items) == 0:
                            continue
                        if not isinstance(type_items[0], dict):
                            continue

                        # Log first type's item keys for diagnostics
                        if items_found == 0:
                            log.info(f"  Sample '{type_key}' item keys: {list(type_items[0].keys())[:15]}")

                        for item in type_items:
                            item_id = str(item.get("id", ""))
                            if not item_id:
                                continue

                            items_found += 1

                            # Created date
                            for dt_key in ["createdDateTime", "CreatedDate", "createdDate", "createdTime"]:
                                if dt_key in item and item[dt_key]:
                                    scanner_created[item_id] = str(item[dt_key])
                                    break

                            # Modified by / configured by
                            for by_key in ["configuredBy", "modifiedBy", "createdBy", "modifiedByUser"]:
                                if by_key in item and item[by_key]:
                                    scanner_modified[item_id] = str(item[by_key])
                                    break

                    log.info(f"  Scanned {items_found} items across {len([k for k,v in target_ws.items() if isinstance(v, list) and v and isinstance(v[0], dict)])} type keys.")
                    log.info(f"  created_date populated for {len(scanner_created)} items.")
                    log.info(f"  modified_by populated for {len(scanner_modified)} items.")

            elif isinstance(scan_result, pd.DataFrame):
                log.info(f"Scanner returned DataFrame: {len(scan_result)} rows, columns: {list(scan_result.columns)}")

                ws_col = next((c for c in ["Workspace Id", "workspaceId", "workspace_id"] if c in scan_result.columns), None)
                if ws_col:
                    scan_result = scan_result[scan_result[ws_col].astype(str) == workspace_id]

                id_col = next((c for c in ["Id", "id"] if c in scan_result.columns), None)
                created_col = next((c for c in ["Created Date", "createdDateTime", "Created Date Time"] if c in scan_result.columns), None)
                modified_by_col = next((c for c in ["Modified By", "modifiedBy", "configuredBy", "Configured By"] if c in scan_result.columns), None)

                if id_col:
                    for _, row in scan_result.iterrows():
                        item_id = str(row[id_col]) if pd.notna(row.get(id_col)) else None
                        if not item_id:
                            continue
                        if created_col and pd.notna(row.get(created_col)):
                            scanner_created[item_id] = str(row[created_col])
                        if modified_by_col and pd.notna(row.get(modified_by_col)):
                            scanner_modified[item_id] = str(row[modified_by_col])

                log.info(f"  created_date: {len(scanner_created)}, modified_by: {len(scanner_modified)}")
            else:
                log.warning(f"Unexpected return type: {type(scan_result)}")

        except ImportError:
            log.info("semantic-link-labs not installed. Skipping.")
        except Exception as e:
            log.warning(f"Scanner API failed: {e}")
            log.info("Continuing without created_date and modified_by.")

## 9. Merge and enrich

In [ ]:
with timed_section("Merge and enrich"):
    if admin_available and not df_admin.empty:
        admin_name_map = {}
        for ai in admin_items_raw:
            if ai.get("id") in admin_only_ids:
                admin_name_map[ai["id"]] = {
                    "name": ai.get("name", ai.get("displayName", "")),
                    "type": ai.get("type", "Unknown"),
                    "description": ai.get("description", ""),
                }
        df_final = df_core.merge(df_admin, on="id", how="outer")
        for idx, row in df_final[df_final["name"].isna()].iterrows():
            info = admin_name_map.get(row["id"], {})
            for col, val in info.items():
                if val:
                    df_final.at[idx, col] = val
    else:
        df_final = df_core.copy()
        for col in ["created_by", "last_modified", "state", "capacity_id"]:
            if col not in df_final.columns:
                df_final[col] = None

    df_final["created_date"] = df_final["id"].map(lambda x: scanner_created.get(str(x)) if pd.notna(x) else None)
    df_final["modified_by"]  = df_final["id"].map(lambda x: scanner_modified.get(str(x)) if pd.notna(x) else None)
    df_final["workspace_id"]   = workspace_id
    df_final["workspace_name"] = workspace_name

    type_url_map = {
        "Report": "reports", "SemanticModel": "datasets", "Dashboard": "dashboards",
        "Dataflow": "dataflows", "DataPipeline": "pipelines", "Notebook": "notebooks",
        "Lakehouse": "lakehouses", "Warehouse": "warehouses", "SQLEndpoint": "sqlEndpoints",
        "Eventhouse": "eventhouses", "KQLDatabase": "kqlDatabases", "KQLQueryset": "kqlQuerysets",
        "KQLDashboard": "kqlDashboards", "Environment": "environments", "SQLDatabase": "sqlDatabases",
        "MirroredDatabase": "mirroredDatabases", "Eventstream": "eventstreams", "Reflex": "reflexes",
        "CopyJob": "copyJobs", "SparkJobDefinition": "sparkJobDefinitions",
    }
    df_final["web_url"] = df_final.apply(
        lambda r: f"{FABRIC_APP}/groups/{workspace_id}/{type_url_map[r['type']]}/{r['id']}"
        if r.get("type") in type_url_map and r.get("id") else None, axis=1
    )
    df_final["snapshot_id"]       = snapshot_id
    df_final["snapshot_time_utc"] = snapshot_time_utc
    df_final = df_final.sort_values(["type", "name"]).reset_index(drop=True)

    log.info(f"Total items: {len(df_final)}")
    log.info(f"  created_date populated: {df_final['created_date'].notna().sum()}")
    log.info(f"  modified_by populated:  {df_final['modified_by'].notna().sum()}")


## 10. Type summary

In [ ]:
with timed_section("Type summary"):
    summary = df_final.groupby("type").size().reset_index(name="count").sort_values("count", ascending=False).reset_index(drop=True)
    print(f"\nObject counts by type ({len(df_final)} total, {summary['type'].nunique()} types):\n")
    display(summary)


## 11. Unused artifacts + date enrichment

Dates are returned as **epoch milliseconds** — converted via vectorized `pd.to_datetime(unit='ms')`.


In [ ]:
unused_artifact_ids = set()
unused_created_dates = {}
unused_last_accessed = {}

with timed_section("Unused artifacts (semantic-link-labs)"):
    if not enable_unused_artifacts:
        log.info("Disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            import sempy_labs.admin as sll_admin

            log.info("Calling list_unused_artifacts()...")
            df_unused = sll_admin.list_unused_artifacts()

            if df_unused is None or (hasattr(df_unused, 'empty') and df_unused.empty):
                log.info("list_unused_artifacts() returned 0 items.")
            else:
                log.info(f"Returned {len(df_unused)} items. Columns: {list(df_unused.columns)}")

                # ── Find ID column ─────────────────────────
                id_col = next((c for c in ["Artifact Id", "Id", "id", "artifactId"] if c in df_unused.columns), None)

                if id_col:
                    unused_artifact_ids = set(df_unused[id_col].dropna().astype(str))
                    log.info(f"  Matched {len(unused_artifact_ids)} item IDs as unused.")

                # ── Date columns ───────────────────────────
                created_col  = next((c for c in ["Created Date Time", "Created Date", "createdDateTime"] if c in df_unused.columns), None)
                accessed_col = next((c for c in ["Last Accessed Date Time", "Last Accessed", "lastAccessedDateTime"] if c in df_unused.columns), None)

                # ── Diagnostic: log actual dtype and sample values ──
                if created_col:
                    col_data = df_unused[created_col]
                    log.info(f"  '{created_col}' dtype={col_data.dtype}, first value={col_data.iloc[0]} (type={type(col_data.iloc[0]).__name__})")
                if accessed_col:
                    col_data = df_unused[accessed_col]
                    log.info(f"  '{accessed_col}' dtype={col_data.dtype}, first value={col_data.iloc[0]} (type={type(col_data.iloc[0]).__name__})")

                # ── Convert epoch-ms to ISO datetimes ──────
                # Handle int, float, string, or object dtype.
                # Strategy: try direct pd.to_datetime(unit=ms) first,
                # then fall back to pd.to_numeric → pd.to_datetime.
                def convert_epoch_col(col_series):
                    """Convert a column of epoch-ms values to UTC datetimes."""
                    # Attempt 1: direct conversion (works for int/float columns)
                    try:
                        result = pd.to_datetime(col_series, unit="ms", utc=True, errors="coerce")
                        if result.notna().any():
                            return result
                    except (ValueError, TypeError):
                        pass

                    # Attempt 2: force to numeric first (works for string columns)
                    try:
                        numeric = pd.to_numeric(col_series, errors="coerce")
                        result = pd.to_datetime(numeric, unit="ms", utc=True, errors="coerce")
                        if result.notna().any():
                            return result
                    except (ValueError, TypeError):
                        pass

                    # Attempt 3: maybe already ISO strings
                    try:
                        result = pd.to_datetime(col_series, errors="coerce", utc=True)
                        if result.notna().any():
                            return result
                    except (ValueError, TypeError):
                        pass

                    return pd.Series([pd.NaT] * len(col_series))

                if created_col and id_col:
                    dt_vals = convert_epoch_col(df_unused[created_col])
                    converted = dt_vals.notna().sum()
                    log.info(f"  '{created_col}' converted: {converted}/{len(df_unused)}")
                    for aid, dt in zip(df_unused[id_col].astype(str), dt_vals):
                        if pd.notna(dt) and str(aid) != "nan":
                            unused_created_dates[str(aid)] = dt.isoformat()
                    log.info(f"  Extracted created_date for {len(unused_created_dates)} items.")

                if accessed_col and id_col:
                    dt_vals = convert_epoch_col(df_unused[accessed_col])
                    converted = dt_vals.notna().sum()
                    log.info(f"  '{accessed_col}' converted: {converted}/{len(df_unused)}")
                    for aid, dt in zip(df_unused[id_col].astype(str), dt_vals):
                        if pd.notna(dt) and str(aid) != "nan":
                            unused_last_accessed[str(aid)] = dt.isoformat()
                    log.info(f"  Extracted last_accessed for {len(unused_last_accessed)} items.")

                if len(df_unused) > 0:
                    display(df_unused)

        except ImportError:
            log.info("semantic-link-labs not installed.")
        except Exception as e:
            log.warning(f"list_unused_artifacts() failed: {e}")

## 12. Activity Events API

In [ ]:
activity_data = {}

with timed_section("Activity Events API"):
    if not enable_activity_events:
        log.info("Disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            end_dt   = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
            start_dt = end_dt - timedelta(days=min(activity_lookback_days, 30))
            log.info(f"Scanning: {start_dt.date()} → {end_dt.date()} | {POWERBI_API_BASE}/admin/activityevents")

            all_events = []
            current_day = start_dt
            days_scanned = 0
            days_failed = 0
            first_event_logged = False
            first_failure_logged = False

            while current_day < end_dt:
                day_str = current_day.strftime("%Y-%m-%d")
                url = f"{POWERBI_API_BASE}/admin/activityevents?startDateTime='{day_str}T00:00:00.000Z'&endDateTime='{day_str}T23:59:59.000Z'"

                while url:
                    try:
                        data = call_powerbi_api(url, caller="ActivityEvents")
                    except PermissionError:
                        raise
                    except Exception as e:
                        # Full diagnostics for the first failure only
                        if not first_failure_logged:
                            log.warning(f"First Activity Events failure — full diagnostics:")
                            log.warning(f"  Day requested: {day_str}")
                            log.warning(f"  URL: {url}")
                            log.warning(f"  Exception: {e}")
                            try:
                                err_resp = requests.get(url, headers=get_headers())
                                log.warning(f"  Response status: {err_resp.status_code}")
                                log.warning(f"  Response body: {err_resp.text[:500]}")
                                resp_headers = {k: v for k, v in err_resp.headers.items()
                                                if k.lower() in ('x-powerbi-error-info', 'x-ms-error',
                                                                  'retry-after', 'content-type', 'requestid',
                                                                  'x-requestid', 'request-id')}
                                if resp_headers:
                                    log.warning(f"  Relevant headers: {resp_headers}")
                            except Exception:
                                pass
                            first_failure_logged = True
                        days_failed += 1
                        break

                    events = data.get("activityEventEntities", [])
                    all_events.extend(events)

                    if events and not first_event_logged:
                        log.info(f"  Sample event keys: {list(events[0].keys())[:15]}")
                        first_event_logged = True

                    url = data.get("continuationUri")

                current_day += timedelta(days=1)
                days_scanned += 1
                if days_scanned % 10 == 0:
                    log.info(f"  Scanned {days_scanned} days, {len(all_events)} events...")

            log.info(f"Retrieved {len(all_events)} events over {days_scanned} days ({days_failed} skipped).")

            if all_events:
                df_events = pd.DataFrame(all_events)

                id_col   = next((c for c in ["ArtifactId", "artifactId", "ItemId", "itemId"] if c in df_events.columns), None)
                time_col = next((c for c in ["CreationTime", "creationTime", "Timestamp"] if c in df_events.columns), None)
                user_col = next((c for c in ["UserId", "userId", "UserKey"] if c in df_events.columns), None)

                if id_col and time_col:
                    df_events["_item_id"] = df_events[id_col].astype(str)
                    df_events["_time"]    = pd.to_datetime(df_events[time_col], errors="coerce", utc=True)

                    for item_id, grp in df_events.groupby("_item_id"):
                        last_used = grp["_time"].max()
                        activity_data[item_id] = {
                            "last_used":    last_used.isoformat() if pd.notna(last_used) else None,
                            "access_count": len(grp),
                            "unique_users": grp[user_col].nunique() if user_col else None,
                        }
                    log.info(f"Aggregated usage for {len(activity_data)} distinct items.")

        except PermissionError:
            log.warning("Activity Events API requires Fabric Admin. Skipping.")
        except Exception as e:
            log.warning(f"Activity Events failed: {e}")

## 13. Merge usage data

In [ ]:
with timed_section("Merge usage data"):
    # Activity Events
    df_final["last_used_date"]   = df_final["id"].map(lambda x: activity_data.get(str(x), {}).get("last_used") if pd.notna(x) else None)
    df_final["access_count_30d"] = df_final["id"].map(lambda x: activity_data.get(str(x), {}).get("access_count") if pd.notna(x) else None)
    df_final["unique_users_30d"] = df_final["id"].map(lambda x: activity_data.get(str(x), {}).get("unique_users") if pd.notna(x) else None)

    # Unused artifact flag
    if unused_artifact_ids:
        df_final["is_unused_artifact"] = df_final["id"].apply(lambda x: 1 if (pd.notna(x) and str(x) in unused_artifact_ids) else 0)
    else:
        df_final["is_unused_artifact"] = 0

    # Enrich created_date from unused artifacts
    if unused_created_dates:
        df_final["created_date"] = df_final.apply(
            lambda r: r["created_date"] if pd.notna(r.get("created_date")) else unused_created_dates.get(str(r["id"]), None) if pd.notna(r.get("id")) else None,
            axis=1
        )
        log.info(f"created_date after enrichment: {df_final['created_date'].notna().sum()}")

    # Enrich last_used_date from unused artifacts (last access before becoming unused)
    if unused_last_accessed:
        df_final["last_used_date"] = df_final.apply(
            lambda r: r["last_used_date"] if pd.notna(r.get("last_used_date")) else unused_last_accessed.get(str(r["id"]), None) if pd.notna(r.get("id")) else None,
            axis=1
        )
        log.info(f"last_used_date after enrichment: {df_final['last_used_date'].notna().sum()}")

    # Computed days
    now_utc = pd.Timestamp.now(tz="UTC")
    df_final["days_since_last_used"] = (now_utc - pd.to_datetime(df_final["last_used_date"], errors="coerce", utc=True)).dt.days
    df_final["days_since_modified"]  = (now_utc - pd.to_datetime(df_final["last_modified"], errors="coerce", utc=True)).dt.days

    log.info(f"Items with activity events:   {(df_final['access_count_30d'].notna()).sum()}")
    log.info(f"Items with last_used (total):  {df_final['last_used_date'].notna().sum()}")
    log.info(f"Items flagged unused artifact:  {(df_final['is_unused_artifact'] == 1).sum()}")


## 14. Governance analysis

All boolean flags stored as **int (1/0)** — immune to Fabric's bool→NaN serialization bug.


In [ ]:
with timed_section("Governance analysis"):
    # ── 1. Stale (vectorized, int output) ──────────────────
    dsm = df_final["days_since_modified"].fillna(99999)
    dsu = df_final["days_since_last_used"].fillna(99999)
    has_usage = df_final["last_used_date"].notna()

    mod_stale = dsm > stale_cutoff_days
    use_stale = dsu > stale_cutoff_days

    stale_bool = np.where(
        has_usage,
        mod_stale & use_stale,       # has usage → both must be stale
        mod_stale                      # no usage → modification only
    )
    df_final["is_stale"] = pd.array(stale_bool, dtype="int64")
    log.info(f"Stale items (>{stale_cutoff_days}d): {df_final['is_stale'].sum()}")

    # ── 2. Missing owner ───────────────────────────────────
    cb = df_final["created_by"].fillna("").astype(str).str.strip().str.lower()
    df_final["has_missing_owner"] = np.where((cb == "") | (cb == "nan") | (cb == "none"), 1, 0)
    log.info(f"Missing owner: {df_final['has_missing_owner'].sum()}")

    # ── 3. Duplicate names (null names excluded) ───────────
    df_final["is_duplicate_name"] = 0
    has_name = df_final["name"].notna() & (df_final["name"].astype(str).str.strip() != "")
    if has_name.any():
        named = df_final.loc[has_name].copy()
        named["_key"] = named["name"].str.lower().str.strip() + "||" + named["type"].str.lower().str.strip()
        dup_keys = set(named["_key"].value_counts().pipe(lambda s: s[s > 1]).index)
        df_final.loc[has_name, "is_duplicate_name"] = np.where(named["_key"].isin(dup_keys), 1, 0)
    log.info(f"Duplicate name+type: {df_final['is_duplicate_name'].sum()}")

    # ── 4. Orphaned semantic models ────────────────────────
    report_names = set(df_final.loc[df_final["type"] == "Report", "name"].dropna().str.lower().str.strip())
    model_mask = df_final["type"] == "SemanticModel"
    df_final["is_orphaned_model"] = 0
    if model_mask.any():
        df_final.loc[model_mask, "is_orphaned_model"] = np.where(
            ~df_final.loc[model_mask, "name"].str.lower().str.strip().isin(report_names), 1, 0
        )
    log.info(f"Orphaned semantic models: {df_final['is_orphaned_model'].sum()}")

    # ── 5. Orphaned SQL endpoints ──────────────────────────
    parent_names = set(df_final.loc[df_final["type"].isin(["Lakehouse", "Warehouse"]), "name"].dropna().str.lower().str.strip())
    ep_mask = df_final["type"] == "SQLEndpoint"
    df_final["is_orphaned_endpoint"] = 0
    if ep_mask.any():
        df_final.loc[ep_mask, "is_orphaned_endpoint"] = np.where(
            ~df_final.loc[ep_mask, "name"].str.lower().str.strip().isin(parent_names), 1, 0
        )
        orphaned = df_final.loc[df_final["is_orphaned_endpoint"] == 1, "name"].tolist()
        if orphaned:
            log.info(f"  Orphaned endpoint names: {orphaned}")
    log.info(f"Orphaned SQL endpoints: {df_final['is_orphaned_endpoint'].sum()}")

    # ── 6. Cleanup candidate score (0-100) ─────────────────
    df_final["cleanup_candidate_score"] = 0
    df_final.loc[df_final["is_stale"] == 1, "cleanup_candidate_score"] += 30
    df_final.loc[df_final["is_unused_artifact"] == 1, "cleanup_candidate_score"] += 25
    df_final.loc[df_final["has_missing_owner"] == 1, "cleanup_candidate_score"] += 15
    df_final.loc[df_final["is_duplicate_name"] == 1, "cleanup_candidate_score"] += 10
    df_final.loc[(df_final["is_orphaned_model"] == 1) | (df_final["is_orphaned_endpoint"] == 1), "cleanup_candidate_score"] += 10
    df_final.loc[dsm > 180, "cleanup_candidate_score"] += 10

    high = (df_final["cleanup_candidate_score"] >= 50).sum()
    med  = ((df_final["cleanup_candidate_score"] >= 30) & (df_final["cleanup_candidate_score"] < 50)).sum()
    log.info(f"Cleanup scores: {high} high-risk (>=50), {med} medium-risk (30-49)")


## 15. Stale items report

In [ ]:
with timed_section("Stale items report"):
    stale_df = df_final[df_final["is_stale"] == 1].sort_values("cleanup_candidate_score", ascending=False)
    if stale_df.empty:
        print(f"No stale items (threshold: {stale_cutoff_days} days).")
    else:
        cols = [c for c in ["name", "type", "created_by", "days_since_modified", "days_since_last_used", "cleanup_candidate_score"] if c in stale_df.columns]
        print(f"\nStale items (>{stale_cutoff_days}d inactive): {len(stale_df)}\n")
        display(stale_df[cols])


## 16. Top cleanup candidates

In [ ]:
with timed_section("Cleanup candidates"):
    top = df_final[df_final["cleanup_candidate_score"] >= 30].sort_values("cleanup_candidate_score", ascending=False)
    if top.empty:
        print("No items scored >=30.")
    else:
        cols = [c for c in ["name", "type", "created_by", "is_stale", "is_unused_artifact", "has_missing_owner", "is_duplicate_name", "is_orphaned_model", "is_orphaned_endpoint", "cleanup_candidate_score"] if c in top.columns]
        print(f"\nCleanup candidates (score >=30): {len(top)}\n")
        display(top[cols].head(50))


## 17. Full inventory

In [ ]:
print(f"\nFull inventory: {len(df_final)} items\n")
display(df_final)


## 18. Persist to Delta table

In [ ]:
with timed_section("Delta table persistence"):
    if not save_to_lakehouse:
        log.info("Skipping (save_to_lakehouse=False).")
    elif df_final.empty:
        log.warning("No data.")
    else:
        try:
            output_columns = [
                "id", "name", "type", "description", "web_url",
                "workspace_id", "workspace_name", "capacity_id",
                "created_by", "modified_by", "created_date", "last_modified",
                "last_used_date", "access_count_30d", "unique_users_30d", "is_unused_artifact",
                "days_since_modified", "days_since_last_used",
                "state", "is_stale", "has_missing_owner", "is_duplicate_name",
                "is_orphaned_model", "is_orphaned_endpoint", "cleanup_candidate_score",
                "snapshot_id", "snapshot_time_utc",
            ]
            output_columns = [c for c in output_columns if c in df_final.columns]
            df_out = df_final[output_columns].copy()
            spark_df = spark.createDataFrame(df_out.astype(str))
            spark_df.write.mode("append").format("delta").saveAsTable(lakehouse_table_name)
            log.info(f"Saved to {lakehouse_table_name}: {len(df_out)} rows, snapshot {snapshot_id}")
        except Exception as e:
            log.error(f"Could not save: {e}")


## 19. Execution summary

In [ ]:
print("=" * 70)
print("  FABRIC WORKSPACE INVENTORY v3.2 — EXECUTION SUMMARY")
print("=" * 70)
print(f"  Workspace:        {workspace_name} ({workspace_id})")
print(f"  Snapshot ID:      {snapshot_id}")
print(f"  Snapshot time:    {snapshot_time_utc}")
print(f"  Admin access:     {'Yes' if admin_available else 'No'}")
print(f"  Scanner API:      created_date={df_final['created_date'].notna().sum()}, modified_by={df_final['modified_by'].notna().sum()}")
print(f"  Activity events:  {len(activity_data)} items with usage data")
print(f"  Unused artifacts: {len(unused_artifact_ids)} flagged, {len(unused_created_dates)} dates enriched")
print(f"  ")
print(f"  INVENTORY")
print(f"  ─────────")
print(f"  Total items:      {len(df_final)}")
print(f"  Item types:       {df_final['type'].nunique()}")
print(f"  ")
print(f"  METADATA COVERAGE")
print(f"  ─────────────────")
for col in ["id","name","created_by","modified_by","created_date","last_modified","last_used_date","web_url"]:
    if col in df_final.columns:
        n = df_final[col].notna().sum()
        print(f"  {col:25s} {n:4d}/{len(df_final)} ({n/len(df_final)*100:5.1f}%)")
print(f"  ")
print(f"  GOVERNANCE FLAGS")
print(f"  ────────────────")
print(f"  Stale (1):                {(df_final['is_stale']==1).sum()}")
print(f"  Not stale (0):            {(df_final['is_stale']==0).sum()}")
print(f"  Missing owner:            {(df_final['has_missing_owner']==1).sum()}")
print(f"  Duplicate names:          {(df_final['is_duplicate_name']==1).sum()}")
print(f"  Orphaned models:          {(df_final['is_orphaned_model']==1).sum()}")
print(f"  Orphaned SQL endpoints:   {(df_final['is_orphaned_endpoint']==1).sum()}")
print(f"  Unused artifacts:         {(df_final['is_unused_artifact']==1).sum()}")
print(f"  Cleanup score >=50:       {(df_final['cleanup_candidate_score']>=50).sum()}")
print(f"  Cleanup score 30-49:      {((df_final['cleanup_candidate_score']>=30)&(df_final['cleanup_candidate_score']<50)).sum()}")
print(f"  ")
print(f"  TIMING")
print(f"  ──────")
total_time = 0
for s, e in section_times.items():
    total_time += e
    print(f"  {s:45s} {e:6.1f}s")
print(f"  {'─'*52}")
print(f"  {'Total':45s} {total_time:6.1f}s")
if save_to_lakehouse:
    print(f"  Delta table: {lakehouse_table_name}")
print("=" * 70)


## 20. Schema reference

| Column | Source | Description |
|--------|--------|-------------|
| `id` | Core/Admin API | Fabric item ID |
| `name` | Core/Admin API | Display name |
| `type` | Core/Admin API | Canonical Fabric type |
| `description` | Core/Admin API | Description |
| `web_url` | Constructed | Fabric portal link |
| `workspace_id` | Parameter | Workspace GUID |
| `workspace_name` | Workspace API | Workspace name |
| `capacity_id` | Admin API | Capacity GUID |
| `created_by` | Admin API | Creator UPN |
| `modified_by` | Scanner API | Last modifier |
| `created_date` | Scanner API + Unused Artifacts | Creation timestamp |
| `last_modified` | Admin API | Modification timestamp |
| `last_used_date` | Activity Events + Unused Artifacts | Last access timestamp |
| `access_count_30d` | Activity Events | Events in 30 days |
| `unique_users_30d` | Activity Events | Distinct users in 30 days |
| `is_unused_artifact` | semantic-link-labs | 1 if unused by Power BI metrics |
| `days_since_modified` | Computed | Days since last_modified |
| `days_since_last_used` | Computed | Days since last_used_date |
| `state` | Admin API | Item state |
| `is_stale` | Governance | 1 if inactive beyond cutoff, 0 if not |
| `has_missing_owner` | Governance | 1 if no creator |
| `is_duplicate_name` | Governance | 1 if name+type duplicated |
| `is_orphaned_model` | Governance | 1 if SemanticModel has no Report |
| `is_orphaned_endpoint` | Governance | 1 if SQLEndpoint has no parent |
| `cleanup_candidate_score` | Governance | 0-100 risk score |

All boolean flags are **int (1/0)** to prevent Fabric's bool→NaN serialization.
